### **Chapter 6.3: Gaussian Process Learning and GP-based MPC**

Chapter 6.2 built a learning-based MPC around a Bayesian linear regression (BLR) model of the terrain and showed that a probabilistic model enables constraint tightening. Every model learned there was, however, handed a strong hint: the trigonometric basis already encoded that the terrain is periodic with a frequency close to the true one. This chapter asks two questions that follow naturally from that observation.

- **What does the structure buy, and what does it cost when it is wrong?** We compare the structured BLR model against a **Gaussian process (GP)**, the standard black-box choice for learning robot dynamics, on data with a **gap**: a stretch of terrain the car never visited. Both models return a mean and a variance, but they place their prior in very different places, and the difference only becomes visible where there is no data. We also look at what happens when the structural assumption is misspecified, and how the assumption can be checked before its variance is trusted.

- **What does the variance actually buy in closed loop?** A learned probabilistic model predicts one step ahead. A predictive controller needs a whole trajectory, so the uncertainty has to be **propagated over the prediction horizon** and converted into constraint tightening. We embed the GP in a stochastic MPC with an **active speed limit**, trace out the trade-off between performance and constraint satisfaction as a function of the confidence level, and show that the tightening is only as good as the variance that drives it.

The plant is the mountain car of Chapter 6.2 on the same bumpy terrain, $h(p) = k\cos(18p)$ with $k = 0.01$. Data is collected everywhere except an unexplored gap in the middle of the domain, so the learned model is confident at the edges and ignorant in the middle, and the controller has to cross the gap while respecting the speed limit.

The content is summarized in the table below.

<style>
  table td:nth-child(2) {
    text-align: left;
  }
</style>

<table border="1" style="border-collapse: collapse;">  <!-- Title Row -->
  <tr>
    <th colspan="2" style="text-align:center">Chapter 6.3</th>
  </tr>

  <!-- Row group 1 -->
  <tr>
    <td rowspan="2">Gaussian Process Model Learning</td>
    <td>Example 1.1: Structured vs. Black-Box Models on Data with a Gap</td>
  </tr>
  <tr>
    <td>Example 1.2: Detecting Misspecification with the Marginal Likelihood</td>
  </tr>

  <!-- Row group 2 -->
  <tr>
    <td rowspan="2">Gaussian Process MPC</td>
    <td>Example 2.1: Certainty-Equivalent Control vs. Constraint Tightening</td>
  </tr>
  <tr>
    <td>Example 2.2: When the Uncertainty Estimate Is Miscalibrated</td>
  </tr>

</table>

<br>

First, we need to set up our Python environment and import relevant packages.

In [ ]:
import sys
import os
import numpy as np
import casadi as ca
import scipy.linalg
import matplotlib.pyplot as plt
from acados_template import AcadosModel, AcadosOcp, AcadosOcpSolver

sys.path.append(os.path.abspath(".."))
from utils.env import *
from utils.simulator import *
from ex2_LQR.lqr_utils import *
from ex5_MPC.mpc_utils import *
from ex6_SysID.SysID_utils import *

FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

<br>

----

### **Experimental Setup**

The real environment is the bumpy terrain of Chapter 6.2 (`case = 3`, $k = 0.01$). Two things change with respect to that chapter.

- **The data has a gap.** Measurements of the terrain height are collected on $p \in [-2, -0.5] \cup [0.5, 2]$ and nowhere in between. The task drives the car from $p = -0.5$ to $p = 0.5$, straight through the unexplored region.

- **The speed limit is active.** In Chapter 6.2 the velocity bound was $|v| \le 4$ and the closed-loop trajectory peaked at $|v| \approx 0.77$, so no state constraint was ever active and constraint tightening had nothing to do. Here we impose $v \le v_{\max} = 0.6$, which the nominal controller would like to violate. Everything the tightening does is then visible in the trajectory.

- **Reference Arguments:**

    - Task: $\boldsymbol{x}_0 = [-0.5, 0]^\top$, $\boldsymbol{x}_T = [0.5, 0]^\top$, horizon $N = 20$, control frequency `10` Hz, `8` s of simulation.

    - Constraints: $p \in [-2, 2]$, $v \in [-0.6, 0.6]$, $a \in [-8, 8]$.

    - Costs: $\boldsymbol{Q} = \text{diag}([5,5])$, $R = 0.1$, $\boldsymbol{Q}_f = \boldsymbol{Q}$.

    - Data: `150` samples on $p \in [-2,-0.5] \cup [0.5, 2]$, zero-mean Gaussian measurement noise with standard deviation $\sigma$ as `0.005`, no data on $p \in [-0.5, 0.5]$.

In [ ]:
# Real environment
case_real = 3
terrain_param = 0.01

# Task
initial_position, initial_velocity = -0.5, 0.0
target_position, target_velocity = 0.5, 0.0
x0 = np.array([initial_position, initial_velocity])
xT = np.array([target_position, target_velocity])

# Constraints. The speed limit is the constraint of interest: the nominal controller
# would like to travel at about 0.77 m/s, so 0.6 m/s is active and must be respected.
v_max = 0.6
state_lbs = np.array([-2.0, -v_max])
state_ubs = np.array([ 2.0,  v_max])
input_lbs, input_ubs = -8.0, 8.0

# Controller
N = 20
Q = np.diag([5, 5])
R = np.array([[0.1]])
Qf = Q

# Simulation
freq = 10
t_terminal = 8

# Data collection: measurements everywhere except an unexplored gap
gap = (-0.5, 0.5)
num_samples = 150
sigma_meas = 0.005

# The true system, used only to simulate the plant
env_real = Env(case_real, x0, xT, param=terrain_param, state_lbs=state_lbs,
               state_ubs=state_ubs, input_lbs=input_lbs, input_ubs=input_ubs)
dynamics_real = Dynamics(env_real)

# Collect the dataset once; both parts of this chapter use the same data
np.random.seed(49) # for reproducibility
data_gen = GenerateData(p_range=[(-2.0, gap[0]), (gap[1], 2.0)], num_samples=num_samples,
                        case=case_real, param=terrain_param)
data_gen.set_noise(mean=0.0, std=sigma_meas) # (I.I.D.) zero-mean Gaussian noise
p_train, h_train = data_gen.generate_data()
true_func = data_gen.get_symbolic_function()

p_test = np.linspace(-2.0, 2.0, 801).reshape(-1, 1)
in_gap = (p_test.ravel() > gap[0]) & (p_test.ravel() < gap[1])

data_gen.plot(title="Terrain measurements with a gap on $p \\in [-0.5, 0.5]$")

<br>

----

### **Part 1: Gaussian Process Model Learning**

#### **Example 1.1: Structured vs. Black-Box Models on Data with a Gap**

Both models are Bayesian and both return a mean and a variance, but they place their prior in very different places.

**Bayesian linear regression** puts the prior on a finite set of coefficients,

$$
h(p) = \boldsymbol{\phi}(p)^\top \boldsymbol{\theta}, \qquad \boldsymbol{\theta} \sim \mathcal{N}(\boldsymbol{\mu}_0, \boldsymbol{\Sigma}_0),
$$

so the predictive variance at a test point, $\boldsymbol{\phi}(p)^\top \boldsymbol{\Sigma}_{\boldsymbol{\theta}} \boldsymbol{\phi}(p)$, depends on the data only through $\boldsymbol{\Sigma}_{\boldsymbol{\theta}}$. Once the data pins down the coefficients *anywhere*, the model is confident *everywhere* the basis is valid.

**Gaussian process regression** puts the prior directly on the function, with no basis at all,

$$
h(p) \sim \mathcal{GP}\big(0,\; k(p, p')\big), \qquad k(p, p') = \sigma_f^2 \exp\!\left(-\frac{(p - p')^2}{2\ell^2}\right),
$$

and the posterior variance at a test point grows back towards the prior $\sigma_f^2$ as soon as that point is more than a few lengthscales $\ell$ away from any data. Its confidence is *local*. The GP has no basis to get wrong, but its two hyperparameters $\ell$ (how fast the function may vary) and $\sigma_f$ (how large it may be) carry all of its structural knowledge, and they have to be chosen. The principled choice is to maximize the **log marginal likelihood** of the data, which we do here by grid search.

To expose the difference between the two priors we fit four models to exactly the same data and compare them over the full range $p \in [-2, 2]$, including the gap:

- **BLR, correct basis:** $\boldsymbol{\phi}(p) = [1, \sin 18p, \cos 18p]^\top$, which contains the true terrain.

- **BLR, incorrect basis:** $\boldsymbol{\phi}(p) = [1, \sin 12p, \cos 12p]^\top$, a periodic structure with the wrong frequency. No choice of coefficients can reproduce the terrain, so the model error is **bias**, not noise.

- **GP, tuned:** squared-exponential kernel, $\ell$ and $\sigma_f$ maximizing the log marginal likelihood on the training data.

- **GP, untuned:** the same kernel with hyperparameters chosen without looking at the data, $\ell = 0.5$ (about an eighth of the domain) and $\sigma_f = 0.01$ (of the order of the terrain height).

We judge the models on two separate questions:

- Is the **mean** right? Measured by the RMS prediction error against the true terrain.

- Is the **error bar** right? The models predict a distribution for a *new measurement*, so we test the bands against new noisy observations of the terrain, $h_{\text{obs}} = h_{\text{true}} + \varepsilon$ with the same noise level as the training data, and measure the normalized error $z = (h_{\text{obs}} - \mu) / \sigma$. A well-calibrated model has $\mathrm{RMS}(z) \approx 1$ and a 2-sigma coverage close to `95.4%`. If $\mathrm{RMS}(z) \gg 1$ the model is **overconfident**, its bounds are too tight, and any constraint tightening built on them is unsafe. If $\mathrm{RMS}(z) \ll 1$ the model is **over-conservative**, its bounds are valid but so wide that the resulting MPC will refuse to move.

Throughout the chapter the plotted bands are the **predictive** standard deviation, which contains both the uncertainty about the terrain itself and the measurement noise $\sigma$.

In [ ]:
def trig_basis(freq):
    '''Basis phi(p) = [1, sin(freq p), cos(freq p)] for a single frequency.'''
    return [lambda p: np.ones_like(p),
            lambda p, k=freq: np.sin(k * p),
            lambda p, k=freq: np.cos(k * p)]


def predictive(model, p_test):
    '''Predictive mean and standard deviation of a new measurement (model uncertainty plus noise).'''
    mean, std = model.predict(p_test)          # Identifier_BLR and Identifier_GP both include the noise term
    return mean.ravel(), std.ravel()


def plot_model(model, p_test, true_func, ax, gap=None, title=None, legend=True,
               ylim=(-0.03, 0.03), xlim=(-1.0, 1.0)):
    '''Draw one model's mean and 1/2/3-sigma bands. Identical for BLR and GP, so the
    panels of the comparison can be read against each other directly.'''
    mean, std = predictive(model, p_test)
    p_flat = p_test.ravel()

    if gap is not None:
        ax.axvspan(gap[0], gap[1], color='gray', alpha=0.15, zorder=0)
    ax.plot(p_flat, [float(true_func(p)) for p in p_flat], 'b--', linewidth=2, label='True function')
    ax.plot(model.p_train, model.h_train, 'k.', alpha=0.5, markersize=6, label='Training data')
    ax.plot(p_flat, mean, color='red', linewidth=2, label='Model prediction')
    for i in range(1, 4):
        ax.fill_between(p_flat, mean - i*std, mean + i*std, color='coral',
                        alpha=0.15 if i == 3 else 0.3,
                        label='Model uncertainty (3 std.)' if i == 3 else None)
    ax.set_xlabel('p'); ax.set_ylabel('h(p)'); ax.grid(True)
    if ylim is not None:
        ax.set_ylim(*ylim)
    if xlim is not None:
        ax.set_xlim(*xlim)
    if title:
        ax.set_title(title, fontsize=11)
    if legend:
        ax.legend(fontsize=8, loc='lower left')


def calibration_report(models, labels, p_test, true_func, gap, noise_std, seed=7):
    '''
    Compare the error a model actually makes against the error bar it claims.

    The RMS error is measured against the true terrain. The band is tested against new
    noisy observations h_obs = h_true + noise: with z = (h_obs - mean) / std, a
    well-calibrated predictive model has RMS(z) close to 1 and covers 95.4 % of the
    observations with +-2 std. RMS(z) > 1 means overconfident (bounds too tight, so
    constraint tightening based on them is unsafe); RMS(z) < 1 means over-conservative.
    '''
    h_true = np.array([float(true_func(p)) for p in p_test.ravel()])
    h_obs = h_true + np.random.default_rng(seed).normal(0.0, noise_std, h_true.shape)
    in_gap = (p_test.ravel() > gap[0]) & (p_test.ravel() < gap[1])

    print(f"{'model':<28}{'region':<12}{'RMS error':>11}{'mean std':>11}{'RMS z':>8}{'2-sigma cov.':>14}")
    print("-" * 84)
    for model, label in zip(models, labels):
        mean, std = predictive(model, p_test)
        for region_name, mask in (("with data", ~in_gap), ("in the gap", in_gap)):
            err = h_true[mask] - mean[mask]
            z = (h_obs[mask] - mean[mask]) / std[mask]
            print(f"{label:<28}{region_name:<12}{np.sqrt(np.mean(err**2)):>11.5f}"
                  f"{np.mean(std[mask]):>11.5f}{np.sqrt(np.mean(z**2)):>8.2f}"
                  f"{100*np.mean(np.abs(z) < 2):>13.1f}%")
    print("-" * 84)
    print(f"RMS amplitude of the true terrain inside the gap: {np.sqrt(np.mean(h_true[in_gap]**2)):.5f} "
          f"(a model predicting only zero scores exactly this); measurement noise std: {noise_std}.")
    print("Nominal 2-sigma coverage of new observations for a calibrated model: 95.4%.")

In [ ]:
# Structured models: BLR with the correct and with an incorrect periodic basis
model_blr_right = Identifier_BLR(trig_basis(18), sigma2=sigma_meas**2)
model_blr_right.fit(p_train, h_train)
model_blr_wrong = Identifier_BLR(trig_basis(12), sigma2=sigma_meas**2)
model_blr_wrong.fit(p_train, h_train)

# Black-box models: GP with hyperparameters tuned by marginal likelihood, and untuned
model_gp = Identifier_GP(noise_std=sigma_meas)
(l_opt, sf_opt), lml_opt = model_gp.optimize_hyperparameters(p_train, h_train)
model_gp.fit(p_train, h_train)
print(f"GP hyperparameters from marginal likelihood: lengthscale = {l_opt:.4f}, signal std = {sf_opt:.5f}")

model_gp_untuned = Identifier_GP(lengthscale=0.5, signal_std=0.01, noise_std=sigma_meas)
model_gp_untuned.fit(p_train, h_train)

models_11 = [model_blr_right, model_blr_wrong, model_gp, model_gp_untuned]
labels_11 = ["BLR, correct basis", "BLR, incorrect basis", "GP, tuned", "GP, untuned"]
titles_11 = ["BLR, correct basis\n$\\phi = [1, \\sin 18p, \\cos 18p]^\\top$",
             "BLR, incorrect basis\n$\\phi = [1, \\sin 12p, \\cos 12p]^\\top$",
             f"GP, tuned\n$\\ell = {l_opt:.2f}$, $\\sigma_f = {sf_opt:.4f}$",
             f"GP, untuned\n$\\ell = {model_gp_untuned.lengthscale:.2f}$, $\\sigma_f = {model_gp_untuned.signal_std:.4f}$"]

_, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True, sharey=True)
for ax, model, title in zip(axes.ravel(), models_11, titles_11):
    plot_model(model, p_test, true_func, ax, gap=gap, title=title, legend=(ax is axes[0, 0]))
plt.tight_layout()
plt.show()

calibration_report(models_11, labels_11, p_test, true_func, gap, sigma_meas)

#### **Results Analysis**

**The correct structure generalizes across the gap; the black-box model does not.** Inside the gap the BLR model with the correct basis reaches an RMS error of about `0.0004`, the same as where it has data: the terrain never had to be observed at $p = 0$ for the model to know what it looks like, because the periodic basis lets data collected on both sides determine amplitude and phase in the middle. The tuned GP reaches an RMS error of about `0.0070` in the gap, which is essentially the RMS amplitude of the terrain itself (`0.0069`): the score of a model that predicts nothing but zero. The GP has reverted to its prior mean and learned nothing about the unvisited stretch. It knows this, however: its predictive band widens from `0.0056` where there is data to `0.0087` in the gap, and both models keep their nominal coverage there (`98%`). Both bands are honest, but they are not equally useful. The BLR band in the gap is the measurement noise and nothing else, because the model has essentially no uncertainty left about the terrain; the GP band carries an extra `0.007` of terrain uncertainty, which propagated through constraint tightening demands margin in exactly the region where the car has no data and most needs to act.

**A wrong basis fails silently, which is the worst way to fail.** With the frequency `12` the model cannot represent the terrain at all: its RMS error is about `0.0071` even in the region densely covered by data, no better than predicting zero. Yet its predictive band there is `0.00505`, identical to that of the correctly specified model. The result is $\mathrm{RMS}(z) \approx 1.7$ and a 2-sigma coverage of about `73%` against the nominal `95.4%`: the model is not merely wrong, it is wrong while claiming the same precision it would have if it were right. The reason is structural: the BLR posterior covariance $\boldsymbol{\Sigma}_{\boldsymbol{\theta}} = ( \boldsymbol{\Sigma}_0^{-1} + \sigma^{-2}\boldsymbol{\Phi}^\top\boldsymbol{\Phi} )^{-1}$ depends on the regressor matrix and the assumed noise variance, but **never on the residuals**. The model computes its error bars from how much data it has and where, not from how well it fits, and the measurement-noise term then hides a bias that is larger than the noise itself. Collecting more data does not help: the error is bias, and more samples only shrink $\boldsymbol{\Sigma}_{\boldsymbol{\theta}}$ further.

**The black-box bounds are only as good as their calibration.** The untuned GP smooths straight through the oscillation with its long lengthscale and reports a band of `0.0051` where it has plenty of data, barely above the noise: $\mathrm{RMS}(z) \approx 1.7$ and a coverage of `73%`, the same silent failure as the wrong basis. In the gap its mean drifts to an error of `0.013`, twice the amplitude of the terrain, against a band of `0.0064`, so it is overconfident there as well ($\mathrm{RMS}(z) \approx 2$, coverage `63%`). Only the marginal-likelihood setting is calibrated ($\mathrm{RMS}(z) \approx 0.95$, coverage `95%` to `98%` everywhere), and finding it required an optimization over the training set. The structured model needs no such tuning: its uncertainty is inherited from the posterior over three coefficients, and it stays calibrated as long as the basis is right.

The practical reading for learning-based control is not that GPs are a poor choice. It is that a black-box model buys freedom from modelling assumptions at the price of a calibration step, and that the calibration must be trusted before its variance is used to tighten a constraint. Where genuine structural knowledge exists, encoding it in the model buys both a better mean and a band that stays at the noise level even where no data exist. Where it does not, a tuned GP degrades gracefully: it was never the best model in any region, but it was never dangerously wrong either.

<br>

----

#### **Example 1.2: Detecting Misspecification with the Marginal Likelihood**

Example 1.1 showed that the posterior variance of a BLR model cannot tell a correct basis from a wrong one, and that the GP hyperparameters decide whether its variance means anything. Both problems have the same remedy. The **log marginal likelihood**, or evidence,

$$
\log p(\boldsymbol{h} \mid \boldsymbol{p}) = -\tfrac{1}{2}\, \boldsymbol{r}^\top \boldsymbol{C}^{-1} \boldsymbol{r} - \tfrac{1}{2} \log\det \boldsymbol{C} - \tfrac{D}{2}\log 2\pi,
$$

with $\boldsymbol{C} = \sigma^2 \boldsymbol{I} + \boldsymbol{\Phi}\boldsymbol{\Sigma}_0\boldsymbol{\Phi}^\top$ and $\boldsymbol{r} = \boldsymbol{h} - \boldsymbol{\Phi}\boldsymbol{\mu}_0$ for BLR, and $\boldsymbol{C} = \boldsymbol{K} + \sigma^2 \boldsymbol{I}$ for the GP, is the probability of the observed data under the model *before* conditioning on it. Unlike the posterior variance it does depend on the residuals, through the first term, and it penalizes needless flexibility through the second. It can therefore rank structural assumptions against each other: different bases for BLR, different hyperparameters for the GP.

We compute the evidence for the four models of Example 1.1 and, for the GP, map it over the $(\ell, \sigma_f)$ plane to see where the tuned and the untuned hyperparameters sit.

In [ ]:
def blr_log_marginal_likelihood(model, p, h):
    '''
    Evidence of the data under a BLR model: h ~ N(Phi mu0, sigma^2 I + Phi Sigma0 Phi^T).

    Unlike the posterior variance, this quantity does depend on the residuals, so it can
    tell us that a basis is wrong even though the error bars cannot.
    '''
    p = np.asarray(p).reshape(-1, 1)
    h = np.asarray(h).reshape(-1, 1)
    Phi = np.hstack([f(p) for f in model.basis_functions])
    B = Phi.shape[1]
    mu0 = model.mu0_user if model.mu0_user is not None else np.zeros((B, 1))
    Sigma0 = model.Sigma0_user if model.Sigma0_user is not None else np.eye(B)

    C = model.sigma2 * np.eye(len(p)) + Phi @ Sigma0 @ Phi.T
    L = np.linalg.cholesky(C)
    r = h - Phi @ mu0
    alpha = np.linalg.solve(L.T, np.linalg.solve(L, r))
    return float(-0.5 * (r.T @ alpha).item() - np.sum(np.log(np.diag(L))) - 0.5 * len(p) * np.log(2 * np.pi))


# Evidence of the four models
evidence = [blr_log_marginal_likelihood(model_blr_right, p_train, h_train),
            blr_log_marginal_likelihood(model_blr_wrong, p_train, h_train),
            model_gp.log_marginal_likelihood(p_train, h_train),
            model_gp_untuned.log_marginal_likelihood(p_train, h_train)]
print(f"{'model':<28}{'log evidence':>14}")
print("-" * 42)
for label, lml in zip(labels_11, evidence):
    print(f"{label:<28}{lml:>14.1f}")
print("-" * 42)
print("Higher is better.")

# Map of the GP evidence over the hyperparameter plane
lengthscales = np.logspace(-2, np.log10(2.0), 50)
signal_stds = np.logspace(np.log10(5e-4), -1, 40)
lml_grid = np.zeros((len(signal_stds), len(lengthscales)))
probe = Identifier_GP(noise_std=sigma_meas)
for i, sf in enumerate(signal_stds):
    for j, l in enumerate(lengthscales):
        probe.lengthscale, probe.signal_std = l, sf
        lml_grid[i, j] = probe.log_marginal_likelihood(p_train, h_train)

_, ax = plt.subplots(figsize=(8, 5))
levels = np.linspace(lml_opt - 400, lml_opt, 21)
cs = ax.contourf(lengthscales, signal_stds, np.maximum(lml_grid, lml_opt - 400), levels=levels, cmap='viridis')
plt.colorbar(cs, ax=ax, label='log marginal likelihood (clipped below)')
ax.plot(l_opt, sf_opt, 'w*', markersize=14, markeredgecolor='k', label='tuned (maximum)')
ax.plot(model_gp_untuned.lengthscale, model_gp_untuned.signal_std, 'rs', markersize=9,
        markeredgecolor='k', label='untuned')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel(r'lengthscale $\ell$'); ax.set_ylabel(r'signal std $\sigma_f$')
ax.set_title('GP evidence over the hyperparameter plane')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

#### **Results Analysis**

**The bound cannot detect misspecification, but the evidence can.** The correct basis scores a log evidence of about `553`, the incorrect basis about `409`, even though the two reported identical error bars in Example 1.1. The residual term in the evidence sees what the posterior covariance cannot. The tuned GP sits in between at about `535`: its evidence is lower than that of the correct basis, because a smooth prior over all functions is a far less specific hypothesis than "a sinusoid at frequency 18", but well above the wrong basis and the untuned GP (about `435`).

**The tuned hyperparameters are a genuine optimum, and the untuned ones are far from it.** The evidence map shows a single ridge: short lengthscales are needed to follow the oscillation, and the signal standard deviation is pinned to the amplitude of the terrain. The untuned point ($\ell = 0.5$, $\sigma_f = 0.01$) lies about `100` nats below the maximum. Nothing about the untuned values was unreasonable a priori. The point is that a black-box model has no other channel through which the data can correct its structure, so the optimization is not optional.

This suggests the practical discipline for learning-based control. The posterior variance answers *"given that my model class is right, how sure am I?"*, and it must not be used to answer *"is my model class right?"*. That second question needs a separate check, such as the marginal likelihood, held-out residuals, or cross-validation, and it should be answered before the variance is ever used to tighten a constraint. Part 2 shows what happens in closed loop when it is not.

<br>

----

### **Part 2: Gaussian Process MPC**

#### **The Learned Model and the Quantity the Controller Needs**

We now take the tuned GP of Part 1 into closed loop. To embed it in the optimal control problem the model is converted into CasADi expressions, exactly as the BLR model was in Chapter 6.2.

The controller needs three things from the model, and the third is easy to get wrong. The dynamics

$$
\dot p = v, \qquad \dot v = u\cos\theta(p) - g\sin\theta(p)\cos\theta(p)
$$

depend on the terrain **only through the inclination** $\theta(p) = \arctan\big(h'(p)\big)$. What propagates into the process noise is therefore the uncertainty of the *slope*, $\mathrm{Var}[h'(p)]$, and not the uncertainty of the height, $\mathrm{Var}[h(p)]$. These are different objects, and the first cannot be recovered from the second: it is the second mixed derivative of the posterior covariance,

$$
\mathrm{Var}[h'(p)] = \frac{\partial^2}{\partial p \, \partial p'} \, \mathrm{Cov}\big[h(p), h(p')\big] \Big|_{p' = p},
$$

which for the squared-exponential kernel evaluates to $\sigma_f^2/\ell^2 - (\partial_p k_*)^\top K^{-1} (\partial_p k_*)$. Differentiating the marginal variance with respect to $p$ would give a different, signed quantity. `construct_gp_casadi_expression` returns all three functions: the posterior mean $h(p)$, the height variance $\mathrm{Var}[h(p)]$, and the slope variance $\mathrm{Var}[h'(p)]$.

In [ ]:
# Convert the tuned GP to CasADi so the model can be used inside the MPC
h_gp, sigma2_h_gp, sigma2_dh_gp = construct_gp_casadi_expression(model_gp)

# Inspect the two quantities the controller needs
std_slope = np.array([np.sqrt(float(sigma2_dh_gp(p))) for p in p_test.ravel()])
true_slope = -terrain_param * 18 * np.sin(18 * p_test.ravel())

_, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_model(model_gp, p_test, true_func, axes[0], gap=gap, title="Learned terrain $h(p)$ (tuned GP)",
           ylim=(-0.05, 0.05), xlim=(-2, 2))

axes[1].plot(p_test, true_slope, 'b--', linewidth=2, label="true slope $h'(p)$")
axes[1].plot(p_test, 3 * std_slope, color='firebrick', linewidth=2, label=r"$3\sqrt{\mathrm{Var}[h'(p)]}$")
axes[1].plot(p_test, -3 * std_slope, color='firebrick', linewidth=2)
axes[1].axvspan(gap[0], gap[1], color='gray', alpha=0.15, zorder=0)
axes[1].set_xlabel("p"); axes[1].set_ylabel("slope"); axes[1].grid(True); axes[1].legend()
axes[1].set_title("Slope uncertainty, which drives the constraint tightening")
plt.tight_layout(); plt.show()

print(f"slope std where data is dense: {np.sqrt(float(sigma2_dh_gp(-1.2))):.4f}")
print(f"slope std inside the gap     : {np.sqrt(float(sigma2_dh_gp(0.0))):.4f}")

#### **Stochastic MPC with a Learned Model**

Because the learned model predicts one step at a time, the controller has to carry a state distribution forward itself. Starting from the measured state, with $\boldsymbol{\Sigma}^x_{0|k} = \boldsymbol{0}$, we roll the mean forward through the learned dynamics and propagate the covariance by linearization,

$$
\boldsymbol{\Sigma}^x_{i+1|k} = \boldsymbol{A}_i \, \boldsymbol{\Sigma}^x_{i|k} \, \boldsymbol{A}_i^\top + \boldsymbol{\Sigma}^w\!\left(\boldsymbol{\mu}^x_{i|k}, \boldsymbol{u}_{i|k}\right),
$$

where $\boldsymbol{A}_i$ is the discrete-time Jacobian of the learned dynamics and $\boldsymbol{\Sigma}^w$ is obtained by pushing $\mathrm{Var}[h'(p)]$ through the dynamics. The first term makes uncertainty accumulate along the horizon; the second injects fresh model uncertainty at each step, and it is large exactly where the GP has no data.

The chance constraint $\mathbb{P}\big(\boldsymbol{x}_{i|k} \in \mathcal{X}\big) \ge \alpha$ is then replaced by the deterministic tightening

$$
\underline{\boldsymbol{x}} + \beta \sqrt{\mathrm{diag}\,\boldsymbol{\Sigma}^x_{i|k}} \;\le\; \boldsymbol{\mu}^x_{i|k} \;\le\; \overline{\boldsymbol{x}} - \beta \sqrt{\mathrm{diag}\,\boldsymbol{\Sigma}^x_{i|k}},
$$

with $\beta$ selected from the confidence level under a Gaussian assumption ($\beta = 2$ is roughly $95\%$ per constraint). Setting $\beta = 0$ recovers a **certainty-equivalent** controller that uses the learned mean and ignores its uncertainty.

The state constraints are implemented as soft constraints with a large penalty. This is deliberate: an over-tightened problem then remains solvable and reports a violation, rather than failing to solve and leaving us unable to distinguish "too conservative" from "broken".

Two implementation details matter enough to be explicit about. First, the propagation uses an **ancillary feedback**: the open-loop linearization of the car on a bump has eigenvalues of roughly $\pm 5.6$, so an open-loop recursion $\boldsymbol{A}_i\boldsymbol{\Sigma}_i\boldsymbol{A}_i^\top$ amplifies the covariance by about four orders of magnitude over a 2 s horizon and predicts nothing useful. The predicted state is not open loop, since the controller keeps acting, so we propagate under $\boldsymbol{A}_i + \boldsymbol{B}_i\boldsymbol{K}$ with $\boldsymbol{K}$ an LQR gain, as is standard in tube- and covariance-based MPC. Second, the tightened box is never allowed to shrink below a fixed fraction of its original width, and the number of times that cap binds is reported, so that an unattainable confidence level is visible rather than silently clipped.

In [ ]:
# The controller is implemented in ex6_SysID/gpmpc_utils.py and shared with Chapter 6.4.
from ex6_SysID.gpmpc_utils import GPMPCController

<br>

----

#### **Example 2.1: Certainty-Equivalent Control vs. Constraint Tightening**

We run the same controller at $\beta \in \{0, 1, 2, 3\}$ against the real plant, and include as a reference the same controller given the true terrain and zero model uncertainty. That reference is the best achievable performance under this speed limit, not a safety benchmark; it rides the limit exactly, and the small overshoot it records is the soft-constraint penalty trading a negligible violation against cost.

In [ ]:
def make_learned_env(h_func, sigma2_h_func, sigma2_dh_func):
    '''Build an Env/Dynamics pair that the controller believes in.'''
    env_l = Env(case_real, x0, xT, symbolic_h_mean_ext=h_func, symbolic_h_cov_ext=sigma2_h_func,
                symbolic_dh_cov_ext=sigma2_dh_func, param=terrain_param, state_lbs=state_lbs,
                state_ubs=state_ubs, input_lbs=input_lbs, input_ubs=input_ubs)
    return env_l, Dynamics(env_l)


def closed_loop_cost(states, inputs):
    err = np.asarray(states) - xT
    return float(np.sum(np.einsum('ij,jk,ik->i', err, Q, err)) + R[0, 0] * np.sum(np.asarray(inputs)**2))


def run_closed_loop(env_l, dynamics_l, beta, name):
    '''Simulate the controller, which trusts env_l, against the real plant.'''
    controller = GPMPCController(env_l, dynamics_l, Q, R, Qf, N, freq, beta=beta, name=name, verbose=False)
    sim = Simulator(dynamics_real, controller, env_real, 1/freq, t_terminal)
    sim.run_simulation()
    states, inputs = np.array(sim.state_traj), np.array(sim.input_traj)
    v = states[:, 1]
    return dict(states=states, inputs=inputs, sigma=np.array(controller.Sigma_x_log),
                peak_v=v.max(), violation=max(0.0, v.max() - v_max),
                steps_over=int((v > v_max + 1e-6).sum()),
                cost=closed_loop_cost(states, inputs), n_saturated=controller.n_saturated,
                final_error=abs(states[-1, 0] - target_position))


def report(rows):
    print(f"{'controller':<32}{'peak v':>9}{'violation':>11}{'steps>lim':>11}{'cost':>9}{'final err':>11}{'sat.':>7}")
    print("-" * 90)
    for label, r in rows:
        print(f"{label:<32}{r['peak_v']:>9.4f}{r['violation']:>11.4f}{r['steps_over']:>11d}"
              f"{r['cost']:>9.1f}{r['final_error']:>11.4f}{r['n_saturated']:>7d}")
    print("-" * 90)
    print(f"speed limit v_max = {v_max}")

In [ ]:
# Performance ceiling: the same controller, but given the true terrain and no model uncertainty
p_sym_zero = ca.MX.sym("p")
zero_var = ca.Function("zero_var", [p_sym_zero], [ca.MX(0.0)])
env_true = Env(case_real, x0, xT, symbolic_h_cov_ext=zero_var, symbolic_dh_cov_ext=zero_var,
               param=terrain_param, state_lbs=state_lbs, state_ubs=state_ubs,
               input_lbs=input_lbs, input_ubs=input_ubs)
res_true = run_closed_loop(env_true, Dynamics(env_true), 0.0, "MPC_true_model")

# GP-MPC at increasing confidence levels
betas = [0.0, 1.0, 2.0, 3.0]
res_gp = {}
for b in betas:
    env_l, dyn_l = make_learned_env(h_gp, sigma2_h_gp, sigma2_dh_gp)
    res_gp[b] = run_closed_loop(env_l, dyn_l, b, f"GPMPC_beta{int(b)}")

report([("MPC, true model (ceiling)", res_true)] +
       [(f"GP-MPC, beta = {b:.0f}" + ("  (certainty equiv.)" if b == 0 else ""), res_gp[b]) for b in betas])

In [ ]:
t = np.arange(len(res_true['states'])) / freq
beta_colors = ['tab:red', 'tab:orange', 'tab:green', 'tab:blue']

_, axes = plt.subplots(1, 3, figsize=(17, 4))

# Closed-loop velocity against the speed limit
axes[0].axhline(v_max, color='k', linestyle='--', linewidth=1.5, label='speed limit')
axes[0].plot(t, res_true['states'][:, 1], color='0.5', linewidth=1.5, label='true model')
for b, c in zip(betas, beta_colors):
    axes[0].plot(t, res_gp[b]['states'][:, 1], color=c, linewidth=1.8, label=rf'GP-MPC $\beta$={b:.0f}')
axes[0].set_xlim(0, 4)
axes[0].set_xlabel('time [s]'); axes[0].set_ylabel('velocity $v$'); axes[0].grid(True)
axes[0].legend(fontsize=8); axes[0].set_title('Closed-loop velocity (first 4 s)')

# Uncertainty growth along the prediction horizon
sig = res_gp[2.0]['sigma']
for k, style in ((0, '-'), (len(sig)//3, '--'), (2*len(sig)//3, ':')):
    axes[1].plot(np.arange(N + 1), np.sqrt(sig[k][:, 1]), style, linewidth=2,
                 label=f'at $t$ = {k/freq:.1f} s')
axes[1].set_xlabel('prediction step $i$'); axes[1].set_ylabel(r'$\sqrt{\Sigma^x_{i|k}}$ (velocity)')
axes[1].grid(True); axes[1].legend(fontsize=8)
axes[1].set_title('Uncertainty propagated over the horizon')

# Performance / safety trade-off
axes[2].plot([res_gp[b]['violation'] for b in betas], [res_gp[b]['cost'] for b in betas],
             'o-', color='tab:blue', linewidth=1.8)
for b in betas:
    axes[2].annotate(rf'$\beta$={b:.0f}', (res_gp[b]['violation'], res_gp[b]['cost']),
                     textcoords='offset points', xytext=(6, 4), fontsize=9)
axes[2].plot(res_true['violation'], res_true['cost'], '*', color='0.4', markersize=14, label='true model')
axes[2].set_xlabel('constraint violation'); axes[2].set_ylabel('closed-loop cost')
axes[2].grid(True); axes[2].legend(fontsize=8)
axes[2].set_title('Performance versus safety')

plt.tight_layout(); plt.show()

#### **Results Analysis**

**Trusting the learned mean is not safe.** At $\beta = 0$ the controller plans against the GP posterior mean, which reverts towards its prior inside the unexplored gap and so understates the terrain the car actually meets. The closed-loop velocity reaches `0.739` against a limit of `0.6`, a violation of `0.139` sustained over two sampling instants. The model error did not stay in the model: it became a constraint violation.

**Tightening converts uncertainty into safety, at a price.** As $\beta$ increases the violation shrinks and the closed-loop cost rises monotonically: `0.139` at cost `96.6`, then `0.065` at `98.1`, then zero at `101.6`, then zero at `108.4`. The constraint becomes satisfied between $\beta = 1$ and $\beta = 2$, and the cost of buying that safety is about `5%` over the certainty-equivalent controller. Going on to $\beta = 3$ costs a further `7%` and buys nothing, since the constraint was already satisfied. This curve, not any single run, is the design object: the confidence level is a knob with a measurable price, and the price of the *first* useful amount of safety is small.

**Uncertainty accumulates along the horizon.** The middle panel shows $\sqrt{\Sigma^x_{i|k}}$ for the velocity growing from zero at the measured state to about `0.085` at the end of the horizon early in the run, and to about `0.5` once the predicted trajectory crosses the data gap. A model that is accurate one step ahead can still produce a wide distribution twenty steps out, and it is the far end of the horizon that sets how hard the constraints must be tightened.

**The tightening sometimes asks for more room than exists.** The `sat.` column counts how often the requested margin exceeded the available half-width and was capped: `64` times at $\beta = 1$, rising to `557` at $\beta = 3$, out of $80 \times 20 = 1600$ tightened constraints. Where the predicted uncertainty approaches the size of the constraint set itself, the nominal confidence level is simply not attainable, and the achievable $\beta$ is limited by the horizon length. Reporting this is preferable to clipping it silently, because it distinguishes "this controller is conservative" from "this confidence level was never delivered".

**The learned model still tracks poorly.** Every GP-MPC run ends `0.28` short of the target, against `0.14` for the true model. This is the mean model failing, not the uncertainty machinery: the GP has no information in the gap and its mean flattens there, so the car settles where it believes the terrain is level. Uncertainty quantification makes the controller *safe* despite a poor model; it does not make the model good.

<br>

----

#### **Example 2.2: When the Uncertainty Estimate Is Miscalibrated**

Example 2.1 used the GP whose hyperparameters were tuned in Part 1. In practice they are often fixed offline, because re-optimizing them online is expensive. We now repeat the experiment with the **untuned GP of Example 1.1**: the same data and the same kernel, but $\ell = 0.5$ and $\sigma_f = 0.01$, chosen without inspecting the data.

The mean of this model is visibly worse, but that is not the point. The point is what it reports about its own reliability, and what the constraint tightening then does with that report.

In [ ]:
# The untuned GP of Example 1.1, converted to CasADi
h_un, var_un, dvar_un = construct_gp_casadi_expression(model_gp_untuned)

print(f"slope std inside the gap:  tuned {np.sqrt(float(sigma2_dh_gp(0.0))):.4f}   "
      f"untuned {np.sqrt(float(dvar_un(0.0))):.4f}   "
      f"(the untuned model claims {np.sqrt(float(sigma2_dh_gp(0.0)))/np.sqrt(float(dvar_un(0.0))):.0f}x less uncertainty)\n")

res_un = {}
for b in (2.0, 3.0):
    env_u, dyn_u = make_learned_env(h_un, var_un, dvar_un)
    res_un[b] = run_closed_loop(env_u, dyn_u, b, f"GPMPCun_beta{int(b)}")

report([(f"GP-MPC tuned,   beta = {b:.0f}", res_gp[b]) for b in (2.0, 3.0)] +
       [(f"GP-MPC UNTUNED, beta = {b:.0f}", res_un[b]) for b in (2.0, 3.0)])

#### **Results Analysis**

**A miscalibrated variance cannot be repaired by tightening.** The untuned GP reports a slope standard deviation of about `0.008` inside the gap where the tuned model reports `0.097`: it claims roughly thirteen times less uncertainty than it has. The tightening is computed faithfully from that number, so it is thirteen times too small. At $\beta = 2$ the closed loop violates the speed limit by `0.102`, and at $\beta = 3$ it still violates by `0.084`. Raising $\beta$ moves the margin only slightly because the margin was computed from the wrong quantity, and note that the untuned controller never once saturates its tightening: it has plenty of room and simply does not ask for it.

Compare the two rows at $\beta = 2$. The tuned model satisfies the constraint at a cost of `101.6`; the untuned model violates it at a cost of `100.2`. The untuned controller looks marginally cheaper precisely because it is buying no safety, which is the trade the numbers would suggest if one ranked controllers by cost alone.

This is the closed-loop consequence of the misspecification studied in Part 1. A model whose predictive variance does not reflect its actual error reports a confident bound, the controller tightens by that bound, and the constraint is still violated. The failure is silent from inside the loop: the optimizer converges, and the tightened constraints are satisfied by the *predicted* trajectory. Only the real system disagrees.

The practical consequence is that the confidence level and the calibration of the model are not interchangeable knobs. $\beta$ trades performance for margin **given** a trustworthy variance. It cannot compensate for a variance that is wrong, and raising it in response to observed violations hides the real problem while paying for conservatism. The evidence check of Example 1.2 would have flagged this model before it was ever put in the loop.

<blockquote style="
    padding: 18px 20px;
    margin: 1.2em 0;
    background: rgba(56, 139, 253, 0.12);      
    border-left: 4px solid rgba(56, 139, 253, 0.85);
    border-radius: 6px;
    color: inherit !important;                  
">

##### **Takeaway:**

- **Structure is leverage, and leverage cuts both ways.** A structured model (BLR with a correct basis) generalizes across unexplored regions with a band that stays at the noise level. A misspecified structure fails silently: its posterior variance never sees the residuals, so it is confidently wrong. A black-box GP is never the best model, but with tuned hyperparameters it is never dangerously wrong either.

- **The variance answers the wrong question about the model class.** Whether the structure or the hyperparameters are right must be checked separately, for example with the **marginal likelihood**, before the variance is used for control.

- A learned probabilistic model predicts **one step**; a predictive controller needs the uncertainty **propagated over the horizon**, and the far end of the horizon is what sets the required margin. What propagates is the uncertainty of the quantity that actually enters the dynamics, here the terrain **slope** $\mathrm{Var}[h'(p)]$, not the height.

- **Constraint tightening turns uncertainty into safety at a measurable cost.** The confidence level $\beta$ traces out a performance-versus-safety curve, and choosing a point on that curve is the design decision.

- **Tightening is only as good as the variance driving it.** A model that understates its uncertainty produces a confident bound, a satisfied prediction, and a violated constraint. Calibration must be established before the variance is trusted, and $\beta$ is not a substitute for it.

</blockquote>

<br>

----

### **Appendix: Figures for the Tutorial Paper**

The two cells below redraw the results of Examples 1.1 and 2.1 at single-column width for the tutorial paper and save them to `figures/`. They contain no new computation.

In [ ]:
PAPER_RC = {"font.size": 7, "axes.titlesize": 7, "axes.labelsize": 7,
            "xtick.labelsize": 6, "ytick.labelsize": 6, "legend.fontsize": 6,
            "axes.linewidth": 0.6, "lines.linewidth": 1.0,
            "font.family": "serif", "mathtext.fontset": "dejavuserif", "pdf.fonttype": 42}

with plt.rc_context(PAPER_RC):
    fig, axes = plt.subplots(2, 2, figsize=(3.5, 3.0), sharex=True, sharey=True)
    p_flat = p_test.ravel()
    h_true = np.array([float(true_func(p)) for p in p_flat])
    for ax, model, title in zip(axes.ravel(), models_11, titles_11):
        mean, std = predictive(model, p_test)
        ax.axvspan(*gap, color="0.85", zorder=0, linewidth=0)
        for j in (3, 2, 1):
            ax.fill_between(p_flat, mean - j*std, mean + j*std, color="lightcoral", alpha=0.15,
                            linewidth=0, zorder=2, label=r"$\pm 1,2,3\,\sigma_h$" if j == 3 else None)
        ax.plot(p_train, h_train, ".", color="0.35", markersize=1.4, alpha=0.7, zorder=3, label="data")
        ax.plot(p_flat, mean, "-", color="firebrick", zorder=4, label=r"$\hat h(p)$")
        ax.plot(p_flat, h_true, "--", color="tab:blue", linewidth=0.9, zorder=5, label=r"$h(p)$")
        ax.set_title(title, pad=3)
        ax.set_ylim(-0.028, 0.028); ax.set_xlim(-1.0, 1.0)
        ax.set_yticks([-0.02, 0, 0.02]); ax.set_xticks([-1, -0.5, 0, 0.5, 1])
        ax.tick_params(length=2, pad=1.5)
    for ax in axes[1]:
        ax.set_xlabel("$p$")
    for ax in axes[:, 0]:
        ax.set_ylabel("$h$")
    handles, labels = axes[0, 0].get_legend_handles_labels()
    order = [labels.index(l) for l in (r"$h(p)$", r"$\hat h(p)$", "data", r"$\pm 1,2,3\,\sigma_h$")]
    fig.legend([handles[i] for i in order], [labels[i] for i in order], loc="lower center", ncol=4,
               frameon=False, handlelength=1.4, columnspacing=1.2, handletextpad=0.4, bbox_to_anchor=(0.5, -0.02))
    fig.tight_layout(pad=0.3, h_pad=0.8, w_pad=0.6, rect=[0, 0.055, 1, 1])
    fig.savefig(os.path.join(FIG_DIR, "tutorial_model_learning.pdf"))
    fig.savefig(os.path.join(FIG_DIR, "tutorial_model_learning.png"), dpi=300)
    plt.show()

In [ ]:
RAMP = ["#f0a08c", "#dd6b52", "#c0392b", "#7b241c"]

with plt.rc_context(PAPER_RC):
    fig = plt.figure(figsize=(3.5, 3.3))
    gs = fig.add_gridspec(2, 2, height_ratios=[1.15, 1.0], hspace=0.72, wspace=0.42)

    # (a) closed-loop velocity
    ax = fig.add_subplot(gs[0, :])
    ax.axhline(v_max, color="k", linestyle="--", linewidth=0.9, label=r"limit $v_{\max}$")
    ax.plot(t, res_true['states'][:, 1], color="0.55", linewidth=0.9, label="true model")
    for b, c in zip(betas, RAMP):
        ax.plot(t, res_gp[b]['states'][:, 1], color=c, linewidth=1.1, label=rf"$\beta={b:.0f}$")
    ax.set_xlim(0, 3.5); ax.set_ylim(-0.15, 1.08)
    ax.set_xlabel("time [s]"); ax.set_ylabel("$v$")
    ax.set_title("Closed-loop velocity", pad=3)
    ax.grid(True, linewidth=0.4, alpha=0.5)
    ax.legend(ncol=3, frameon=False, handlelength=1.3, columnspacing=0.9,
              handletextpad=0.4, borderpad=0.1, loc="upper center")

    # (b) uncertainty propagated over the horizon
    ax = fig.add_subplot(gs[1, 0])
    for k, style, lab in ((0, "-", "$t=0$"), (12, "--", "$t=1.2$ s")):
        ax.plot(np.arange(N + 1), np.sqrt(sig[k][:, 1]), style, color="#c0392b", linewidth=1.1, label=lab)
    ax.set_xlabel("prediction step $i$"); ax.set_ylabel(r"$\sigma^v_{i|k}$")
    ax.set_title("Uncertainty growth", pad=3)
    ax.grid(True, linewidth=0.4, alpha=0.5)
    ax.legend(frameon=False, handlelength=1.3, handletextpad=0.4, borderpad=0.1)

    # (c) performance versus safety
    ax = fig.add_subplot(gs[1, 1])
    ax.plot([res_gp[b]['violation'] for b in betas], [res_gp[b]['cost'] for b in betas],
            "-", color="0.6", linewidth=0.8, zorder=1)
    for b, c in zip(betas, RAMP):
        ax.plot(res_gp[b]['violation'], res_gp[b]['cost'], "o", color=c, markersize=4, zorder=2)
        ax.annotate(rf"$\beta={b:.0f}$", (res_gp[b]['violation'], res_gp[b]['cost']),
                    textcoords="offset points", xytext=(4, 3), fontsize=6, color=c)
    ax.axvline(0.0, color="k", linestyle=":", linewidth=0.7)
    ax.set_xlim(-0.03, 0.185)
    costs = [res_gp[b]['cost'] for b in betas]
    ax.set_ylim(min(costs) - 3, max(costs) + 6)
    ax.set_xlabel("constraint violation"); ax.set_ylabel("closed-loop cost")
    ax.set_title("Performance vs. safety", pad=3)
    ax.grid(True, linewidth=0.4, alpha=0.5)

    fig.savefig(os.path.join(FIG_DIR, "tutorial_gp_mpc.pdf"), bbox_inches="tight")
    fig.savefig(os.path.join(FIG_DIR, "tutorial_gp_mpc.png"), dpi=300, bbox_inches="tight")
    plt.show()